# Multi-Knapsack Optimization with Preference-Based Item Assignment


## Packages and Dependencies

In [1]:
import pandas as pd

from ortools.linear_solver import pywraplp


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\debor\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\debor\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\debor\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\debor\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\debor\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\debor\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\debor\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\debor\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\debor\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\debor\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

## Data Loading

In [2]:
#Bags
Bags = pd.read_csv('bags.csv', sep= ';')

#Items
Items = pd.read_csv('Items.csv', sep= ';')


In [3]:
Items

,ID,Name,Weight (kg),Value (USD),Storage Type,Temperature,Humidity,Notes
0,I1,Frozen Chicken,3.0,25.0,Refrigerated,Cold,Humid,Requires thermal bag
1,I2,Rice,2.0,12.0,Dry,Room,Dry,Suitable for fabric or cardboard bags
2,I3,Liquid Soap,1.5,9.5,Liquid,Room,Humid,Ideal for plastic bags
3,I4,Book,1.0,22.0,Dry,Room,Dry,Avoid humid bags
4,I5,Ice Cream,2.5,18.0,Frozen,Very Cold,Humid,Thermal bag only
5,I6,Tomato,2.0,7.0,Fresh,Cool,Humid,Can go in thermal or plastic bag
6,I7,Panettone,3.0,20.0,Dry,Room,Dry,Fabric or cardboard bag recommended
7,I8,Shampoo,1.2,15.0,Liquid,Room,Humid,Preferably in plastic bag


In [4]:
Bags

,ID,Type,Capacity (kg),Material,Notes,Temperature,Humidity
0,B1,Thermal,10,Insulated,Ideal for cold or humid items,Cold,Humid
1,B1,Thermal,10,Insulated,Ideal for cold or humid items,Cool,Humid
2,B1,Thermal,10,Insulated,Ideal for cold or humid items,Very Cold,Humid
3,B2,Fabric,8,Light cloth,Breathable but not moisture-proof,Cool,Dry
4,B2,Fabric,8,Light cloth,Breathable but not moisture-proof,Room,Dry
5,B3,Plastic,6,Waterproof,Protects against liquids,Room,Dry
6,B3,Plastic,6,Waterproof,Protects against liquids,Cool,Dry
7,B3,Plastic,6,Waterproof,Protects against liquids,Room,Humid
8,B3,Plastic,6,Waterproof,Protects against liquids,Cool,Humid
9,B4,Cardboard,5,Fragile,Unsuitable for humid or heavy items,Room,Dry


## Input Data

This section focuses on preparing and structuring the raw data into the specific format required by the optimization function.

* **weights**: A vector containing the weights of the items.

* **values**: A vector containing the importance (or value) of each item.

* **capacities**: A vector containing the capacity limits of each knapsack.

### Bags

In [5]:
bags_dict = {}
for b_id, group in Bags.groupby("ID"):
    first_row = group.iloc[0]
    
    bags_dict[b_id] = {
        "Type": first_row["Type"],
        "Capacity (kg)": int(first_row["Capacity (kg)"]),
        "Material": first_row["Material"],
        "Notes": first_row["Notes"],
        "Temperature": list(group["Temperature"].unique()),
        "Humidity": list(group["Humidity"].unique())
    }

In [6]:
bags = list(Bags['ID'].unique())
print(bags)

['B1', 'B2', 'B3', 'B4']


In [7]:
bags_capacity = dict(zip(Bags['ID'],Bags['Capacity (kg)']))
print(bags_capacity)

{'B1': 10, 'B2': 8, 'B3': 6, 'B4': 5}


### Items

In [8]:
items = list(Items['ID'].unique())
print(items)

['I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8']


In [9]:
items_weight = dict(zip(Items['ID'],Items['Weight (kg)']))
print(items_weight)

{'I1': 3.0, 'I2': 2.0, 'I3': 1.5, 'I4': 1.0, 'I5': 2.5, 'I6': 2.0, 'I7': 3.0, 'I8': 1.2}


### Item preferences per bag

#### Compatibility
Binary variable:

$$
Compatibility_{i,b} =
\begin{cases}
1, & \text{if item } i \text{ can go in bag } b \\
0, & \text{if item } i \text{ can't go in bag } b
\end{cases}
$$

$$
Compatibility_{i,b} \in \{0,1\},
\quad
\forall i = 1, \dots, m,
\quad
\forall b = 1, \dots, n
$$

In [10]:
values={}
compat = {}
max_value = Items['Value (USD)'].max()

for item in items:
    ##------------------------Items------------------------##
    
    #print(item)
    
    item_ = Items.loc[Items['ID'] == item].iloc[0]
    
    #Item Temperature
    item_temperature = item_['Temperature']
    #print(item_temperature)
    
    #Item Humidity
    item_humidity = item_['Humidity']
    #print(item_humidity)
    
    #Item Value
    item_value = item_['Value (USD)']
    #print(item_value)
    
    for bag in bags:
        ##------------------------Bags------------------------##
        
        #print(bag)
        
        bag_ = bags_dict[bag]
        
        #Bag Temperature
        bag_temperature = bag_['Temperature']
        #print(bag_temperature)

        #Bag Humidity
        bag_humidity = bag_['Humidity']
        #print(bag_humidity)
        
        #----------------------------Compatibility------------------------------#
        
        compat[(item, bag)] = 1 if ( item_humidity in bag_humidity ) else 0
        
        
        #----------------------------Score------------------------------#
        
        #Temperature
        score_temperature = 1 if item_temperature in bag_temperature else 0
        #print(score_temperature)
              
        #Humidity
        score_humidity = 1 if item_humidity in bag_humidity else 0
                
        score_total =   round((score_temperature + score_humidity + item_value/max_value)/3 , 3 )
        
        values[(item, bag)] = score_total

## Optimization
### Indices

- $i = 1, \dots, m$: items

- $b = 1, \dots, n$: bags

### Parameters

- $w_i$: weight of item $i$

- $v_{ib}$: value of item $i$ assigned to bag $b$

- $c_b$: capacity of bag $b$

### Creating the optimization solver using the OR-Tools SCIP backend

In [11]:
solver = pywraplp.Solver.CreateSolver("SCIP")

### Decision Variables

Binary variable:

$$
x_{i,b} =
\begin{cases}
1, & \text{if item } i \text{ will be assigned to bag } b \\
0, & \text{otherwise}
\end{cases}
$$

$$
x_{i,b} \in \{0,1\},
\quad
\forall i = 1, \dots, m,
\quad
\forall b = 1, \dots, n
$$

In [12]:
x = {}

for item in items:
    for bag in bags:
        x[item, bag] = solver.BoolVar(f'x[{item},{bag}]')

### Restrictions

#### Capacity restriction
$$
\sum_{i=1}^{m} w_i x_{ib} \leq c_b \quad \forall b=1, \dots,n
$$

In [13]:
for bag in bags:
    solver.Add(sum(x[item, bag] * items_weight[item] for item in items) <= bags_capacity[bag])

#### Item Restriction

Each item can be assigned to a maximum of one bag:

$$
\sum_{b=1}^{n} x_{ib} \leq 1
\quad
\forall i = 1, \dots, m
$$

In [14]:
for item in items:
    solver.Add(sum(x[item, bag] for bag in bags) <= 1)

#### Compatibility Restriction

$$x_{ib} \leq Compatibility_{ib}$$

In [15]:
for item in items:
    for bag in bags:
        solver.Add(x[item, bag] <= compat[(item, bag)])

### Objective Function

Maximize the total value of items assigned to backpacks:

$$
\max
\sum_{i=1}^{m}
\sum_{b=1}^{n}
v_{ib} x_{ib}
$$

In [16]:
solver.Maximize(sum(values[(item, bag)] * x[item, bag] for item in items for bag in bags))

### Solving the Optimization Model

In [17]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    cont_i=0
    print("Total value:", solver.Objective().Value())
    for bag in bags:
        for i, item in enumerate(items):
            if x[item, bag].solution_value() == 1:
                print(f"Item {item} assigned to bag {bag} with value  {values[(item, bag)]}")
                cont_i +=1
    #print(cont_i)                
else:
    print("It was not possible to find an optimal solution.")

Total value: 7.047
Item I1 assigned to bag B1 with value  1.0
Item I5 assigned to bag B1 with value  0.907
Item I6 assigned to bag B1 with value  0.76
Item I2 assigned to bag B2 with value  0.827
Item I4 assigned to bag B2 with value  0.96
Item I7 assigned to bag B2 with value  0.933
Item I3 assigned to bag B3 with value  0.793
Item I8 assigned to bag B3 with value  0.867
